# Research Task - Local Transit Projects List #890
https://github.com/cal-itp/data-analyses/issues/890

Start working on a local (transit) projects list, perhaps by reaching out to people like Jeff Tumlin?
Amanda to start from LRTP lists, Christian to shadow/ask more detailed transit priority questions. "Across CA, how much would we save with transit priority implementation?"

Connected to Julian (student assistant) scope (analyze benefits from list)

Received more clarification. Starting with MPOs LRTP, identify projects that improve existing transit performance (upgrading something, improve ridership, etc). Then compare this short list to the remaining projects in LRTP (% of..., cost vs...). May need to look at previous LRTPs to see if same projects were mentioned previously to see if projects were completed or not.


## Next Steps
Start with reading in Amanda's data from gcs (list of all projects from MPO LRTPs)

In [1]:
import pandas as pd


In [6]:
my_df = pd.read_excel('gs://calitp-analytics-data/data-analyses/project_list/LRTP/all_LRTP_LOST.xlsx')

In [7]:
my_df.shape

(16276, 9)

In [8]:
# filtering out LOST projects
# df.loc[df.data_source != "Lost"] 
my_df = my_df.loc[my_df.data_source != 'Lost']

In [9]:
display(type(my_df))
display(my_df.shape)
display(list(my_df.columns))
display(my_df.head())

pandas.core.frame.DataFrame

(14427, 9)

['project_title',
 'lead_agency',
 'project_year',
 'project_description',
 'total_project_cost',
 'city',
 'county',
 'data_source',
 'notes']

,project_title,lead_agency,project_year,project_description,total_project_cost,city,county,data_source,notes
0,Carmel To Pebble Beach Bike/Ped Facility,Ambag,None,Construct Class I Or Class Ii Bike Facility.,86000.0,None,Santa Cruz,Ambag Lrtp,NaN
1,Sr 1 Carmel Corridor Between Carmel River Brid...,Ambag,None,Provide Accomodation For Bicyclists Along Stat...,500000.0,None,Santa Cruz,Ambag Lrtp,NaN
2,"Rio Road Traffic Calming, Pedestrian Access An...",Ambag,None,"Install Traffic Calming Devices, Enhance Visib...",250000.0,None,Santa Cruz,Ambag Lrtp,NaN
3,Eighth And San Antonio Avenues Class Ii Bike I...,Ambag,None,"Install Signs, Pavement Markings, Intersection...",80000.0,None,Santa Cruz,Ambag Lrtp,NaN
4,Pedestrian Pathway Behind Larson Field And Rio...,Ambag,None,Construct Pedestrian And Possible Bike Route A...,75000.0,None,Santa Cruz,Ambag Lrtp,NaN


## Next
Find Amanda's code snippet of filtering the df by string values. specifically, filtering the projects that are trasnit 
related (either project name/desc/notes)

see these for reference:

https://github.com/cal-itp/data-analyses/blob/2b88d23f3895840cd8714f213ec6a4c1fc55c088/project_list/_categorizing_utils.py#L135

https://github.com/cal-itp/data-analyses/blob/main/project_list/_specific_list_utils.py#L44


In [10]:
transit_keywords = [
        "bus",
        "metro",
        "station",  # Station comes up a few times as a charging station and also as a train station
        "transit",
        "fare",
        "brt",
        "yarts",
        "railroad",
        "highway-rail",
        "streetcar",
        "mass transit",]
        # , 'station' in description and 'charging station' not in description

In [24]:
#test of isin() for transit keywords

test = my_df[my_df['project_title'].str.contains('|'.join(transit_keywords))]
test

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [15]:
#ripping code from Amanda
def lower_case(df, columns_to_search: list):
    """
    Lowercase and clean certain columns:
    """
    new_columns = []
    for i in columns_to_search:
        df[f"lower_case_{i}"] = (
            df[i]
            .str.lower()
            .fillna("none")
            .str.replace("-", "")
            .str.replace(".", "")
            .str.replace(":", "")
        )
        new_columns.append(f"lower_case_{i}")

    return df, new_columns

def find_keywords(df, columns_to_search: list, keywords_search: list):
    """
    Find keywords in certain columns
    """
    df2, lower_case_cols_list = lower_case(df, columns_to_search)

    keywords_search = f"({'|'.join(keywords_search)})"

    for i in lower_case_cols_list:
        df2[f"{i}_keyword_search"] = (
            df2[i].str.extract(keywords_search).fillna("keyword not found")
        )

    return df2

def filter_projects(
    df,
    columns_to_search: list,
    keywords_search: list,
    file_name: str,
    save_to_gcs: bool = True,
):

    # Filter out for Cordon
    df = find_keywords(df, columns_to_search, keywords_search)
    df2 = (
        df[
            (df.lower_case_project_title_keyword_search != "keyword not found")
            | (df.lower_case_project_description_keyword_search != "keyword not found")
        ]
    ).reset_index(drop=True)

    # Delete out non HOV projects that were accidentally picked up
    projects_to_delete = [
        "SR 17 Corridor Congestion Relief in Los Gatos",
        "Interstate 380 Congestion Improvements",
    ]
    df2 = df2[~df2.project_title.isin(projects_to_delete)].reset_index(drop=True)

    # Change cases
    for i in ["project_title", "project_description"]:
        df2[i] = df2[i].str.title()

    # Drop invalid geometries
    #gdf = df2[df2.geometry != None].reset_index(drop=True)
    #gdf = gdf.set_geometry("geometry")
    #gdf.geometry = gdf.geometry.set_crs("EPSG:4326")
    #gdf = gdf[gdf.geometry.is_valid].reset_index(drop=True)
    #gdf = gdf.fillna(gdf.dtypes.replace({"float64": 0.0, "object": "None"}))

    # One version that's a df
    columns_to_drop = ["lower_case_project_title", "lower_case_project_description"]
    #df2 = df2.drop(columns=columns_to_drop + ["geometry"])
    df2 = df2.fillna(df.dtypes.replace({"float64": 0.0, "object": "None"}))

   

    return df2

In [17]:
#ripping amanda's code for find_keywords



In [10]:
# ripping Amanda's code from LRTP project list 



In [16]:
# ripping Amanda's code from LRTP project list 

filter_projects(
    my_df,
    [
        "project_title",
        "project_description",
    ],
    transit_keywords,
    False,
)

/tmp/ipykernel_159/1913973701.py:9: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df[i]


,project_title,lead_agency,project_year,project_description,total_project_cost,city,county,data_source,notes,lower_case_project_title,lower_case_project_description,lower_case_project_title_keyword_search,lower_case_project_description_keyword_search
0,Citywide Sidewalk Improvement Program,Ambag,None,Construct New Sidewalk Per Ada Transition Plan,6000000.0,None,Santa Cruz,Ambag Lrtp,None,citywide sidewalk improvement program,construct new sidewalk per ada transition plan,keyword not found,transit
1,Ada Transition Program,Ambag,None,"City-Wide Sidewalk, Ramp, Intersection, And Bu...",1621000.0,None,Santa Cruz,Ambag Lrtp,None,ada transition program,"citywide sidewalk, ramp, intersection, and bus...",transit,bus
2,Class I Bike Path Along Railroad,Ambag,None,Install Class I Bike Path Along Railroad Row,1300000.0,None,Santa Cruz,Ambag Lrtp,None,class i bike path along railroad,install class i bike path along railroad row,railroad,railroad
3,Ada Transition Plan Upgrades,Ambag,None,Roadway & Sidewalk Improvements,32000000.0,None,Santa Cruz,Ambag Lrtp,None,ada transition plan upgrades,roadway & sidewalk improvements,transit,keyword not found
4,Station Place (Itc Bridge),Ambag,None,Install Bike And Ped Bridge Over Railroad,1500000.0,None,Santa Cruz,Ambag Lrtp,None,station place (itc bridge),install bike and ped bridge over railroad,station,railroad
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1724,V-12 Goshen Trail,Tcag,2035,9 Miles Of Multi-Use Corridor Following Existi...,14000000.0,None,None,Tcag Lrtp,"Project Category: Active Transportation, Juri...",v12 goshen trail,9 miles of multiuse corridor following existin...,keyword not found,railroad
1725,P-5 Butterfield Stage Corridor (Entire Stretch...,Tcag,2035,"Class 1 Multi-Use Path, Solar Lighting, Water ...",7750000.0,None,None,Tcag Lrtp,"Project Category: Active Transportation, Juri...",p5 butterfield stage corridor (entire stretch ...,"class 1 multiuse path, solar lighting, water s...",keyword not found,station
1726,D-04 Ventura Street Pedestrian Path And Railro...,Tcag,2035,"Concrete Sidewalk, Panels For Rail Crossing, H...",847000.0,None,None,Tcag Lrtp,"Project Category: Active Transportation, Juri...",d04 ventura street pedestrian path and railroa...,"concrete sidewalk, panels for rail crossing, h...",railroad,keyword not found
1727,U-13 Goshen-Avenue 308 Improvement,Tcag,2035,"Curb, Gutter, Asphalt Pave-Outs, Bus Stops, Bi...",920000.0,None,None,Tcag Lrtp,"Project Category: Active Transportation, Juri...",u13 goshenavenue 308 improvement,"curb, gutter, asphalt paveouts, bus stops, bik...",keyword not found,bus
